# Statement analytics

**Purpose:** Use `finstack.statements_analytics` to stress-test, compare, explain, and score forecasts on a statement model built with `ModelBuilder`.

**Prerequisites:** Work through **notebook 09** in this track (statement modeling) so you are comfortable building periods, value nodes, compute nodes, and evaluating with `Evaluator`.


## Analytics on statement models

A financial model is a directed graph of periods and nodes. Once you can evaluate it, analytics answers adjacent questions: *What if drivers move?* *How does a new plan differ from the old one?* *What revenue clears a target EBITDA?* *Which scenarios matter?* *Where did this number come from?* *Was the forecast any good?*

The Python bindings expose these workflows as small functions that take JSON (or simple numeric vectors) and return JSON or structured Python values—ideal for notebooks, dashboards, and automated reports.


## Shared baseline: a compact P&L

We build one demo model and serialize it with `spec.to_json()`. Evaluation produces `eval_json` via `result.to_json()`. Later calls reuse these strings so each analytics API stays a pure function of model + configuration.


In [1]:
import json

from finstack.statements import Evaluator, ModelBuilder

b = ModelBuilder("analytics-demo")
b.periods("2025Q1..Q4", "2025Q2")
b.value("revenue", [("2025Q1", 100.0), ("2025Q2", 110.0), ("2025Q3", 115.0), ("2025Q4", 120.0)])
b.value("cogs", [("2025Q1", 60.0), ("2025Q2", 65.0), ("2025Q3", 68.0), ("2025Q4", 72.0)])
b.compute("gross_profit", "revenue - cogs")
b.value("opex", [("2025Q1", 20.0), ("2025Q2", 22.0), ("2025Q3", 23.0), ("2025Q4", 24.0)])
b.compute("ebitda", "gross_profit - opex")
spec = b.build()
model_json = spec.to_json()
result = Evaluator().evaluate(spec)
eval_json = result.to_json()

print("Model id:", json.loads(model_json)["id"])
print("EBITDA 2025Q1 (evaluated):", json.loads(eval_json)["nodes"]["ebitda"]["2025Q1"])


Model id: analytics-demo
EBITDA 2025Q1 (evaluated): 20.0


## Sensitivity and tornado views

`run_sensitivity` perturbs explicit driver nodes and records how target metrics respond. `generate_tornado_entries` narrows that JSON to a ranked list for one metric (and optional period) — useful for plotting.

**Perturbation semantics:** Each entry in `perturbations` is an **absolute replacement value** for the node in that period. The `base_value` field records the original level and is used by `generate_tornado_entries` to classify perturbations as downside (below base) or upside (above base). For example, with `base_value=100` and `perturbations=[-20, ..., 20]`, each value directly replaces revenue — so perturbation `-20` sets revenue to −20, while perturbation `120` would set revenue to 120.


In [2]:
from finstack.statements_analytics import generate_tornado_entries, run_sensitivity

sensitivity_config = json.dumps({
    "mode": "Diagonal",
    "parameters": [{
        "node_id": "revenue",
        "period_id": "2025Q1",
        "base_value": 100.0,
        "perturbations": [-20.0, -10.0, -5.0, 0.0, 5.0, 10.0, 20.0],
    }],
    "target_metrics": ["gross_profit", "ebitda"],
})
sens_result = run_sensitivity(model_json, sensitivity_config)

parsed_sens = json.loads(sens_result)
print(f"Sensitivity: {len(parsed_sens['scenarios'])} scenarios across {parsed_sens['config']['target_metrics']}")
for sc in parsed_sens["scenarios"]:
    pv = list(sc["parameter_values"].values())[0]
    ebitda_q1 = sc["results"]["nodes"]["ebitda"]["2025Q1"]
    print(f"  revenue@Q1 = {pv:6.0f} -> EBITDA@Q1 = {ebitda_q1:8.1f}")

tornado = generate_tornado_entries(sens_result, "ebitda", "2025Q1")
print("\nTornado entries:", json.loads(tornado))


{"config":{"mode":"Diagonal","parameters":[{"node_id":"revenue","period_id":"2025Q1","base_value":100.0,"perturbations":[-20.0,-10.0,-5.0,0.0,5.0,10.0,20.0]}],"target_metrics":["gross_profit","ebitda"]},"scenarios":[{"parameter_values":{"revenue@2025Q1":-20.0},"results":{"nodes":{"opex":{"2025Q1":20.0,"2025Q2":22.0,"2025Q3":23.0,"2025Q4":24.0},"cogs":{"2025Q1":60.0,"2025Q2":65.0,"2025Q3":68.0,"2025Q4":72.0},"revenue":{"2025Q1":-20.0,"2025Q2":110.0,"2025Q3":115.0,"2025Q4":120.0},"gross_profit":{"2025Q1":-80.0,"2025Q2":45.0,"2025Q3":47.0,"2025Q4":48.0},"ebitda":{"2025Q1":-100.0,"2025Q2":23.0,"2025Q3":24.0,"2025Q4":24.0}},"node_value_types":{"revenue":{"type":"scalar"},"cogs":{"type":"scalar"},"gross_profit":{"type":"scalar"},"opex":{"type":"scalar"},"ebitda":{"type":"scalar"}},"meta":{"num_nodes":5,"num_periods":4,"numeric_mode":"float64","parallel":false}}},{"parameter_values":{"revenue@2025Q1":-10.0},"results":{"nodes":{"opex":{"2025Q1":20.0,"2025Q2":22.0,"2025Q3":23.0,"2025Q4":24.0},"

## Variance between two evaluated models

`run_variance` compares two **evaluation** JSON payloads (baseline vs comparison) for selected metrics and periods. Build a second `ModelBuilder`, evaluate it, and pass both `to_json()` outputs plus a `VarianceConfig`.


In [3]:
from finstack.statements_analytics import run_variance

b2 = ModelBuilder("comparison")
b2.periods("2025Q1..Q4", "2025Q2")
b2.value("revenue", [("2025Q1", 105.0), ("2025Q2", 115.0), ("2025Q3", 120.0), ("2025Q4", 125.0)])
b2.value("cogs", [("2025Q1", 62.0), ("2025Q2", 67.0), ("2025Q3", 70.0), ("2025Q4", 74.0)])
b2.compute("gross_profit", "revenue - cogs")
b2.value("opex", [("2025Q1", 21.0), ("2025Q2", 23.0), ("2025Q3", 24.0), ("2025Q4", 25.0)])
b2.compute("ebitda", "gross_profit - opex")
comp_result = Evaluator().evaluate(b2.build())

variance_config = json.dumps({
    "baseline_label": "base",
    "comparison_label": "upside",
    "metrics": ["gross_profit", "ebitda"],
    "periods": ["2025Q1", "2025Q2"],
})
variance = json.loads(run_variance(eval_json, comp_result.to_json(), variance_config))
print(f"Variance: {variance['baseline_label']} vs {variance['comparison_label']}")
for row in variance["rows"]:
    print(f"  {row['period']} {row['metric']:15s}  base={row['baseline']:6.1f}  cmp={row['comparison']:6.1f}  "
          f"abs={row['abs_var']:+.1f}  pct={row['pct_var']:+.1%}")


{"baseline_label":"base","comparison_label":"upside","rows":[{"period":"2025Q1","metric":"gross_profit","baseline":40.0,"comparison":43.0,"abs_var":3.0,"pct_var":0.075},{"period":"2025Q2","metric":"gross_profit","baseline":45.0,"comparison":48.0,"abs_var":3.0,"pct_var":0.06666666666666667},{"period":"2025Q1","metric":"ebitda","baseline":20.0,"comparison":22.0,"abs_var":2.0,"pct_var":0.1},{"period":"2025Q2","metric":"ebitda","baseline":23.0,"comparison":25.0,"abs_var":2.0,"pct_var":0.08695652173913043}]}


## Goal seek

`goal_seek` searches for a driver value so a target node hits a desired level in a given period. With `update_model=False`, the returned model JSON is still updated in the second tuple element per the binding contract—print the solved driver and a short preview of the updated model.


In [4]:
from finstack.statements_analytics import goal_seek

gs_result = goal_seek(
    model_json,
    target_node="ebitda",
    target_period="2025Q1",
    target_value=25.0,
    driver_node="revenue",
    driver_period="2025Q1",
    update_model=False,
)
solved_revenue, updated_model_json = gs_result
print("Solved revenue (2025Q1):", solved_revenue)
print("Updated model JSON (first 400 chars):", updated_model_json[:400])


Solved revenue (2025Q1): 105.0
Updated model JSON (first 400 chars): {
  "id": "analytics-demo",
  "periods": [
    {
      "id": "2025Q1",
      "start": "2025-01-01",
      "end": "2025-04-01",
      "is_actual": true
    },
    {
      "id": "2025Q2",
      "start": "2025-04-01",
      "end": "2025-07-01",
      "is_actual": true
    },
    {
      "id": "2025Q3",
      "start": "2025-07-01",
      "end": "2025-10-01",
      "is_actual": false
    },
    {
     


## Scenario set evaluation

`evaluate_scenario_set` runs every named scenario against the same base model. Each scenario supplies `overrides` as **node id → scalar**; those scalars replace values only in **forecast periods** (periods after `actuals_until`). Actual periods are never overridden — they retain their original values regardless of the override.

In this model, `actuals_until="2025Q2"`, so overrides only affect Q3 and Q4. Below, upside and downside shift `revenue` in the forecast window.


In [5]:
from finstack.statements_analytics import evaluate_scenario_set

scenario_set = json.dumps({
    "scenarios": {
        "upside": {"overrides": {"revenue": 130.0}},
        "downside": {"overrides": {"revenue": 100.0}},
    }
})
scenario_results = json.loads(evaluate_scenario_set(model_json, scenario_set))
for name, sr in scenario_results.items():
    rev = sr["nodes"]["revenue"]
    ebitda = sr["nodes"]["ebitda"]
    print(f"--- {name} ---")
    for p in ["2025Q1", "2025Q2", "2025Q3", "2025Q4"]:
        print(f"  {p}  revenue={rev[p]:6.1f}  ebitda={ebitda[p]:5.1f}")
print("\nNote: Q1–Q2 (actuals) retain original values; Q3–Q4 (forecast) use overrides.")


{"upside":{"nodes":{"opex":{"2025Q1":20.0,"2025Q2":22.0,"2025Q3":23.0,"2025Q4":24.0},"cogs":{"2025Q1":60.0,"2025Q2":65.0,"2025Q3":68.0,"2025Q4":72.0},"revenue":{"2025Q1":100.0,"2025Q2":110.0,"2025Q3":130.0,"2025Q4":130.0},"gross_profit":{"2025Q1":40.0,"2025Q2":45.0,"2025Q3":62.0,"2025Q4":58.0},"ebitda":{"2025Q1":20.0,"2025Q2":23.0,"2025Q3":39.0,"2025Q4":34.0}},"node_value_types":{"revenue":{"type":"scalar"},"cogs":{"type":"scalar"},"gross_profit":{"type":"scalar"},"opex":{"type":"scalar"},"ebitda":{"type":"scalar"}},"meta":{"eval_time_ms":0,"num_nodes":5,"num_periods":4,"numeric_mode":"float64","parallel":false}},"downside":{"nodes":{"opex":{"2025Q1":20.0,"2025Q2":22.0,"2025Q3":23.0,"2025Q4":24.0},"cogs":{"2025Q1":60.0,"2025Q2":65.0,"2025Q3":68.0,"2025Q4":72.0},"revenue":{"2025Q1":100.0,"2025Q2":110.0,"2025Q3":100.0,"2025Q4":100.0},"gross_profit":{"2025Q1":40.0,"2025Q2":45.0,"2025Q3":32.0,"2025Q4":28.0},"ebitda":{"2025Q1":20.0,"2025Q2":23.0,"2025Q3":9.0,"2025Q4":4.0}},"node_value_types

## Dependencies and formula explanation

`trace_dependencies` returns an ASCII tree of upstream nodes. `direct_dependencies` lists immediate inputs. `explain_formula` returns a **Python dict** (JSON-serializable) with the resolved breakdown for one node and period—use `json.dumps` for a stable string view.


In [6]:
from finstack.statements_analytics import direct_dependencies, explain_formula, trace_dependencies

deps = trace_dependencies(model_json, "ebitda")
print("Full dependency tree:")
print(deps)

direct = direct_dependencies(model_json, "ebitda")
print("Direct deps:", direct)

explanation = explain_formula(model_json, eval_json, "ebitda", "2025Q1")
print("Formula explanation (JSON):", json.dumps(explanation, indent=2))


Full dependency tree:
ebitda (gross_profit - opex)
gross_profit (revenue - cogs)
revenue
cogs
opex

Direct deps: ['gross_profit', 'opex']
Formula explanation (JSON): {
  "node_id": "ebitda",
  "period_id": "2025Q1",
  "final_value": 20.0,
  "node_type": "Calculated",
  "formula_text": "gross_profit - opex",
  "breakdown": [
    {
      "component": "gross_profit",
      "value": 40.0,
      "operation": null
    },
    {
      "component": "opex",
      "value": 20.0,
      "operation": null
    }
  ]
}


## Backtest forecast accuracy

`backtest_forecast` expects parallel sequences of actuals and forecasts (same length) and returns MAE, MAPE, RMSE, and count. Here we align two quarters of revenue actuals vs the baseline forecast.


In [7]:
from finstack.statements_analytics import backtest_forecast

actual = [102.0, 108.0]
forecast = [100.0, 110.0]
bt = backtest_forecast(actual, forecast)
print(json.dumps(dict(bt)))


{"mae": 2.0, "mape": 1.9063180827886708, "rmse": 2.0, "n": 2}


## Additional introspection APIs

Beyond `trace_dependencies` and `explain_formula`, the analytics module provides:

- **`trace_dependencies_detailed`** — ASCII tree annotated with values for a specific period
- **`all_dependencies`** — flat list of all transitive upstream nodes
- **`dependents`** — reverse lookup: which nodes depend on a given node?
- **`explain_formula_text`** — human-readable multi-line explanation

In [ ]:
from finstack.statements_analytics import (
    all_dependencies,
    dependents,
    explain_formula_text,
    trace_dependencies_detailed,
)

print("=== trace_dependencies_detailed (ebitda, 2025Q1) ===")
print(trace_dependencies_detailed(model_json, eval_json, "ebitda", "2025Q1"))

print("=== all_dependencies(ebitda) ===")
print(all_dependencies(model_json, "ebitda"))

print("\n=== dependents(revenue) — who uses revenue? ===")
print(dependents(model_json, "revenue"))

print("\n=== explain_formula_text(ebitda, 2025Q1) ===")
print(explain_formula_text(model_json, eval_json, "ebitda", "2025Q1"))

### Pandas export

`StatementResult` also exports to pandas DataFrames (wide and long) alongside the Polars equivalents shown in the statement modeling notebook.

In [ ]:
print("Pandas wide export:")
print(result.to_pandas_wide())
print()
print("Pandas long export (first 8 rows):")
print(result.to_pandas_long().head(8))

## Mini-example: full analytics pass on the demo P&L

The cell below is self-contained: it rebuilds the baseline and comparison models, then runs sensitivity, variance, scenarios, goal seek, dependency tracing, formula explanation, and a tiny revenue backtest—mirroring how you might script a quarterly analytics pack.


In [8]:
import json

from finstack.statements import Evaluator, ModelBuilder
from finstack.statements_analytics import (
    backtest_forecast,
    direct_dependencies,
    evaluate_scenario_set,
    explain_formula,
    generate_tornado_entries,
    goal_seek,
    run_sensitivity,
    run_variance,
    trace_dependencies,
)

# Baseline
b = ModelBuilder("analytics-demo")
b.periods("2025Q1..Q4", "2025Q2")
b.value("revenue", [("2025Q1", 100.0), ("2025Q2", 110.0), ("2025Q3", 115.0), ("2025Q4", 120.0)])
b.value("cogs", [("2025Q1", 60.0), ("2025Q2", 65.0), ("2025Q3", 68.0), ("2025Q4", 72.0)])
b.compute("gross_profit", "revenue - cogs")
b.value("opex", [("2025Q1", 20.0), ("2025Q2", 22.0), ("2025Q3", 23.0), ("2025Q4", 24.0)])
b.compute("ebitda", "gross_profit - opex")
spec = b.build()
model_json = spec.to_json()
base_eval = Evaluator().evaluate(spec).to_json()

# Comparison (upside plan)
b2 = ModelBuilder("comparison")
b2.periods("2025Q1..Q4", "2025Q2")
b2.value("revenue", [("2025Q1", 105.0), ("2025Q2", 115.0), ("2025Q3", 120.0), ("2025Q4", 125.0)])
b2.value("cogs", [("2025Q1", 62.0), ("2025Q2", 67.0), ("2025Q3", 70.0), ("2025Q4", 74.0)])
b2.compute("gross_profit", "revenue - cogs")
b2.value("opex", [("2025Q1", 21.0), ("2025Q2", 23.0), ("2025Q3", 24.0), ("2025Q4", 25.0)])
b2.compute("ebitda", "gross_profit - opex")
cmp_eval = Evaluator().evaluate(b2.build()).to_json()

print("=== Sensitivity (EBITDA tornado, 2025Q1) ===")
sens_cfg = json.dumps({
    "mode": "Diagonal",
    "parameters": [{
        "node_id": "revenue",
        "period_id": "2025Q1",
        "base_value": 100.0,
        "perturbations": [-20.0, -10.0, -5.0, 0.0, 5.0, 10.0, 20.0],
    }],
    "target_metrics": ["gross_profit", "ebitda"],
})
sens = run_sensitivity(model_json, sens_cfg)
print("Tornado:", json.loads(generate_tornado_entries(sens, "ebitda", "2025Q1")))

print("\n=== Variance (Q1–Q2, gross_profit & ebitda) ===")
var_cfg = json.dumps({
    "baseline_label": "base",
    "comparison_label": "upside",
    "metrics": ["gross_profit", "ebitda"],
    "periods": ["2025Q1", "2025Q2"],
})
var_result = json.loads(run_variance(base_eval, cmp_eval, var_cfg))
for row in var_result["rows"]:
    print(f"  {row['period']} {row['metric']:15s}  Δ={row['abs_var']:+.1f} ({row['pct_var']:+.1%})")

print("\n=== Scenarios (revenue shocks, forecast periods only) ===")
scen = json.dumps({
    "scenarios": {
        "upside": {"overrides": {"revenue": 130.0}},
        "downside": {"overrides": {"revenue": 100.0}},
    }
})
scen_out = json.loads(evaluate_scenario_set(model_json, scen))
for name, sr in scen_out.items():
    print(f"  {name}: EBITDA Q3={sr['nodes']['ebitda']['2025Q3']}, Q4={sr['nodes']['ebitda']['2025Q4']}")

print("\n=== Goal seek EBITDA 2025Q1 -> 25 ===")
solved, _mj = goal_seek(
    model_json,
    target_node="ebitda",
    target_period="2025Q1",
    target_value=25.0,
    driver_node="revenue",
    driver_period="2025Q1",
    update_model=False,
)
print("Solved revenue:", solved)

print("\n=== Dependencies for ebitda ===")
print(trace_dependencies(model_json, "ebitda"))
print("Direct:", direct_dependencies(model_json, "ebitda"))
print("Explain:", json.dumps(explain_formula(model_json, base_eval, "ebitda", "2025Q1"), indent=2))

print("\n=== Revenue forecast backtest (2 quarters) ===")
bt = backtest_forecast([102.0, 108.0], [100.0, 110.0])
print(f"MAE={bt['mae']:.2f}  MAPE={bt['mape']:.2f}%  RMSE={bt['rmse']:.2f}  n={bt['n']}")


=== Sensitivity (EBITDA tornado, 2025Q1) ===
[{"parameter_id":"revenue","downside":0.0,"upside":40.0}]
=== Variance (Q1–Q2, gross_profit & ebitda) ===
{"baseline_label":"base","comparison_label":"upside","rows":[{"period":"2025Q1","metric":"gross_profit","baseline":40.0,"comparison":43.0,"abs_var":3.0,"pct_var":0.075},{"period":"2025Q2","metric":"gross_profit","baseline":45.0,"comparison":48.0,"abs_var":3.0,"pct_var":0.06666666666666667},{"period":"2025Q1","metric":"ebitda","baseline":20.0,"comparison":22.0,"abs_var":2.0,"pct_var":0.1},{"period":"2025Q2","metric":"ebitda","baseline":23.0,"comparison":25.0,"abs_var":2.0,"pct_var":0.08695652173913043}]}
=== Scenarios (revenue shocks) ===
{"upside":{"nodes":{"opex":{"2025Q1":20.0,"2025Q2":22.0,"2025Q3":23.0,"2025Q4":24.0},"cogs":{"2025Q1":60.0,"2025Q2":65.0,"2025Q3":68.0,"2025Q4":72.0},"revenue":{"2025Q1":100.0,"2025Q2":110.0,"2025Q3":130.0,"2025Q4":130.0},"gross_profit":{"2025Q1":40.0,"2025Q2":45.0,"2025Q3":62.0,"2025Q4":58.0},"ebitda":{

## Takeaways

- **Sensitivity** (`run_sensitivity`, `generate_tornado_entries`) quantifies how driver shocks flow into key metrics before you commit to a plan.
- **Variance** (`run_variance`) formalizes baseline vs alternate evaluation JSON—ideal for budget vs actual or plan versions.
- **Goal seek** finds the driver level that hits a target metric in a period, preserving the rest of the model structure.
- **Scenarios** (`evaluate_scenario_set`) batch-evaluate named override sets; overrides are node-level scalars applied only to **forecast periods** (periods after `actuals_until`).
- **Introspection** (`trace_dependencies`, `trace_dependencies_detailed`, `all_dependencies`, `dependents`, `explain_formula`, `explain_formula_text`) supports audit trails and IC-ready narrative.
- **Backtesting** (`backtest_forecast`) scores vector forecasts with standard error metrics.
- **Export** to both **Polars** (`to_polars_wide`, `to_polars_long`) and **pandas** (`to_pandas_wide`, `to_pandas_long`).

**Next:** Keep building richer statement models in notebook 09’s spirit, then wire these analytics into your own reporting or scenario libraries.
